# TacticalVision AI — Plug-and-Play Kaggle GPU Worker

Run this notebook on **Kaggle** with:
- **Accelerator:** `GPU T4 x2` or `GPU P100` (30 GB RAM)
- **Internet:** `ON`
- **Attached Dataset:** `tacticalvision-models` (containing the 8 production model weights)

Run Cells **1 → 2 → 3** in order, then copy the printed `https://xxxx.trycloudflare.com` URL into **TacticalVision AI → Analyze Match → Remote GPU (Kaggle / Colab)** and click **Test Connection**.

In [ ]:
# ==============================================================================
# CELL 1: Clone TacticalVision-AI (feat/4-model-integration) & Install Deps
# ==============================================================================
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Mazenalodini/TacticalVision-AI.git"
BRANCH = "feat/4-model-integration"
REPO_DIR = Path("/kaggle/working/TacticalVision-AI")

if REPO_DIR.exists():
    print(f"Updating existing repo at {REPO_DIR}...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
else:
    print(f"Cloning {REPO_URL} ({BRANCH})...")
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)

core_deps = [
    "ultralytics",
    "timm",
    "rapidocr-onnxruntime",
    "imageio-ffmpeg",
    "fastapi",
    "uvicorn",
    "python-multipart",
    "pydantic-settings",
    "httpx",
]
print("Installing lightweight GPU worker dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"] + core_deps, check=True)

# Prefer onnxruntime-gpu on CUDA; fall back cleanly to onnxruntime CPU if wheel mismatch
ort_rc = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "onnxruntime-gpu"],
    check=False,
).returncode
if ort_rc != 0:
    print("onnxruntime-gpu wheel unavailable — installing onnxruntime fallback...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "onnxruntime"], check=True)

print("✅ Cell 1 complete! Working directory:", os.getcwd())


In [ ]:
# ==============================================================================
# CELL 2: Auto-Locate MODELS_DIR under /kaggle/input & Verify GPU + All 8 Models
# ==============================================================================
import os
from pathlib import Path
import torch

EXPECTED_MODELS = [
    ("players_best.pt",),
    ("pitch_best.pt",),
    ("action_spotting_v1_int8.onnx", "action_spotting_v1.onnx"),
    ("action_spotting_v1.pt",),
    ("jersey_legibility_v1_int8.onnx", "jersey_legibility_v1.onnx"),
    ("jersey_legibility_v1.pt",),
    ("resnet152_tf2_backbone.onnx",),
    ("soccernet_pca512_tf2_weights.npz",),
]

def find_models_dir() -> Path:
    search_roots = [Path("/kaggle/input"), Path("/kaggle/working/TacticalVision-AI/models")]
    for root in search_roots:
        if not root.exists():
            continue
        for hit in root.rglob("players_best.pt"):
            return hit.parent
    raise FileNotFoundError(
        "Could not locate 'players_best.pt' under /kaggle/input. "
        "Please attach your 'tacticalvision-models' Kaggle Dataset to this notebook."
    )

MODELS_DIR = find_models_dir()
os.environ["MODELS_DIR"] = str(MODELS_DIR)

print(f"📁 MODELS_DIR = {MODELS_DIR} (read-only mount, zero file copying)")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram = props.total_memory / (1024 ** 3)
    print(f"⚡ CUDA GPU Detected: {torch.cuda.get_device_name(0)} — {vram:.1f} GB VRAM")
else:
    print("⚠️ WARNING: CUDA GPU not detected! Enable GPU T4/P100 in Kaggle Session Options.")

print("\n── 8-Model Production Artifact Checklist ──")
missing = []
for spec in EXPECTED_MODELS:
    found = next((MODELS_DIR / name for name in spec if (MODELS_DIR / name).exists()), None)
    if found is not None:
        size_mb = found.stat().st_size / (1024 ** 2)
        print(f"  ✅ {found.name:<36s} ({size_mb:6.1f} MB)")
    else:
        print(f"  ❌ {spec[0]:<36s} MISSING")
        missing.append(spec[0])

if missing:
    raise RuntimeError(f"Missing required model files in {MODELS_DIR}: {missing}")
print("\n✅ All 8 models verified and ready!")


In [ ]:
# ==============================================================================
# CELL 3: Launch FastAPI GPU Worker (port 8001) + Cloudflare Quick Tunnel
# ==============================================================================
import os
import re
import stat
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

REPO_DIR = Path("/kaggle/working/TacticalVision-AI")
os.chdir(REPO_DIR)

# 1. Download standalone cloudflared binary (zero account / token needed)
cf_bin = Path("/kaggle/working/cloudflared")
if not cf_bin.exists():
    print("Downloading standalone cloudflared binary...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        str(cf_bin),
    )
    cf_bin.chmod(cf_bin.stat().st_mode | stat.S_IEXEC)

# 2. Start uvicorn gpu_worker.server:app on port 8001
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["GPU_WORK_DIR"] = "/kaggle/working/tv_gpu_jobs"

server_log = open("/kaggle/working/gpu_worker_server.log", "w", encoding="utf-8")
uvicorn_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "gpu_worker.server:app", "--host", "0.0.0.0", "--port", "8001"],
    cwd=str(REPO_DIR),
    stdout=server_log,
    stderr=subprocess.STDOUT,
    env=env,
)

# Wait for local /health to come online
ready = False
for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8001/health", timeout=2) as r:
            if r.status == 200:
                print("✅ Local GPU Worker Health:", r.read().decode("utf-8"))
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    uvicorn_proc.terminate()
    raise RuntimeError("uvicorn failed to start! Check /kaggle/working/gpu_worker_server.log")

# 3. Launch Cloudflare Quick Tunnel and print the live URL prominently
tunnel_proc = subprocess.Popen(
    [str(cf_bin), "tunnel", "--url", "http://127.0.0.1:8001", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

public_url = None
url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

try:
    assert tunnel_proc.stdout is not None
    for line in tunnel_proc.stdout:
        m = url_pattern.search(line)
        if m and public_url is None:
            public_url = m.group(0)
            print("\n" + "=" * 72)
            print("🚀 TACTICALVISION AI KAGGLE GPU WORKER IS LIVE!")
            print(f"🔗 Copy & Paste URL:  {public_url}")
            print("=" * 72)
            print("Keep this cell running while analyzing videos from your local studio.")
            print("Stop this cell anytime to shut down the GPU worker cleanly.\n")
except KeyboardInterrupt:
    print("Shutting down Kaggle GPU Worker and Cloudflare Tunnel...")
finally:
    tunnel_proc.terminate()
    uvicorn_proc.terminate()
    server_log.close()
